# Kangaroo-256 build + smoke test (Colab T4)

Verifies that the GPU solver builds and recovers a known key on a small
synthetic puzzle. Run this once before pointing the solver at the real target.

**Prereq:** project (`bitcoin-prize`) must be reachable at `PROJECT_DIR` below.
Easiest paths: `git clone` the repo, upload it via the Files panel, or mount
Drive and set `PROJECT_DIR` to its location there.

**Runtime:** Runtime → Change runtime type → T4 GPU.


In [ ]:
# Sanity: confirm a T4 is attached.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader


In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/bitcoin-prize"   # change to your Drive path if needed
KANGAROO_DIR = "/content/Kangaroo-256"
WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isdir(PROJECT_DIR), (
    f"project not found at {PROJECT_DIR!r}. Either git-clone, upload via "
    f"Files, or set PROJECT_DIR to a Drive path."
)
sys.path.insert(0, PROJECT_DIR)
print("project:", PROJECT_DIR)


In [ ]:
# Clone + build Kangaroo-256. ccap=75 = T4 (compute capability 7.5).
if not os.path.isdir(KANGAROO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/ZenulAbidin/Kangaroo-256.git", KANGAROO_DIR],
        check=True,
    )

subprocess.run(["make", "clean"], cwd=KANGAROO_DIR, check=False,
               capture_output=True)
build = subprocess.run(
    ["make", "gpu=1", "ccap=75", "all"],
    cwd=KANGAROO_DIR, capture_output=True, text=True,
)
print(build.stdout[-2000:])
if build.returncode != 0:
    print("STDERR:", build.stderr[-2000:])
    raise RuntimeError(f"build failed (rc={build.returncode})")

binary = os.path.join(KANGAROO_DIR, "kangaroo-256")
assert os.path.isfile(binary), f"binary missing after build: {binary}"
print("built:", binary)


In [ ]:
# Generate a small synthetic puzzle with a known answer.
# 70 bits → ~70 G ops → ~2 min on T4 at ~600 MK/s.
# Solve time scales as sqrt(N), so +2 bits ≈ 2× wall time.
PUZZLE_PATH = f"{WORK_DIR}/puzzle.txt"
BITS = 70
SEED = 42

gen = subprocess.run(
    ["python3", "scripts/make_synthetic_puzzle.py",
     "--bits", str(BITS), "--seed", str(SEED), "--out", PUZZLE_PATH],
    cwd=PROJECT_DIR, capture_output=True, text=True, check=True,
)
print(gen.stderr.strip())
print("---")
print(open(PUZZLE_PATH).read())


In [ ]:
# Run Kangaroo-256 GPU mode against the synthetic puzzle.
# -d 14 forces a sane DP rate (auto-pick has high variance on narrow ranges).
# -o writes the recovered key to a file; the live status lines on stdout can
# clobber the Priv: line if we relied on parsing stdout alone.
import time, json

RESULT_PATH = "/tmp/k256_result.txt"
if os.path.exists(RESULT_PATH):
    os.remove(RESULT_PATH)

with open(PUZZLE_PATH + ".json") as f:
    manifest = json.load(f)
print(f"expecting d = 0x{manifest['d_hex']}")

t0 = time.monotonic()
solve = subprocess.run(
    [binary, "-t", "1", "-gpu", "-gpuId", "0",
     "-d", "14",
     "-o", RESULT_PATH,
     PUZZLE_PATH],
    capture_output=True, text=True, timeout=900,
)
elapsed = time.monotonic() - t0
print(solve.stdout[-2000:])
print(f"--- elapsed {elapsed:.2f}s, rc={solve.returncode}")


In [ ]:
# Verify: parse Priv: line and compare to manifest.
# Prefer the result file (clean) over stdout (cluttered with status lines).
import re

if os.path.exists(RESULT_PATH):
    out = open(RESULT_PATH).read()
    print(f"--- {RESULT_PATH} ---")
    print(out)
else:
    print(f"{RESULT_PATH} missing; falling back to stdout")
    out = solve.stdout

m = re.search(r"Priv\s*:\s*(?:0x)?([0-9a-fA-F]+)", out)
assert m, f"no Priv: line\n--- last 500 chars ---\n{out[-500:]}"
recovered = m.group(1).lower().lstrip("0") or "0"
expected = manifest["d_hex"].lstrip("0") or "0"
assert recovered == expected, f"key mismatch: got {recovered}, expected {expected}"
print(f"PASS — recovered d = 0x{recovered}")


## What this proves

- The Linux/CUDA build path on Colab T4 works (`ccap=75`).
- Kangaroo-256 produces correct output on the JLP-format input we generate.
- The toolchain — clone → build → puzzle gen → solve → verify — is reproducible.

**Next:** build the production-target notebook (`solver.ipynb`) on top of this:
add Drive mount, work-file checkpointing every 20 min, idle-timeout-clean exit.
